<div align="center">
  <img src="assets/Day10.png" alt="Databricks 14 Days AI Challenge - Day 10" width="800"/>
</div>

## DAY 10 (18/01/26) – Performance Optimization

### 📚 Learning Objectives
In the cloud, **Time = Money**. A query that scans 1TB of data costs 100x more than a query that scans 10GB. Today, we learn how to make Spark "lazy but smart" by skipping unnecessary data:
* **Explain Plans:** Reading the map Spark creates before running code.
* **Partitioning:** Organizing files into folders (Coarse-grained skipping). 
* **Z-Ordering:** Organizing data *inside* the files (Fine-grained skipping). 
* **Caching:** Storing hot data in RAM.

### 🚀 Strategy: The "Speed Raceway"
1.  **Baseline:** Run a specific filter query on our standard Silver table and measure the time.
2.  **Diagnose:** Use `explain()` to see that Spark is performing a "Full Table Scan" (reading everything).
3.  **Optimize:** Create a new table strategy using **Partitioning** (by Date) and **Z-Ordering** (by User/Product).
4.  **Race:** Benchmark the optimized table against the baseline.

###Setup & Baseline Analysis
**Task**: Analyze the query plan. **Concept**: explain() shows the DAG. We are looking for "PushedFilters". If Spark doesn't know where the data is, it reads everything.

In [0]:
import time
from pyspark.sql.functions import col

# Setup context
catalog = "course_catalog"
schema = "ecommerce_governed"
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")

# 1. THE BAD QUERY
# We want to find all actions by a specific user.
# Without optimization, Spark opens every file to check for 'user_id = 563509121'.
target_user = 563509121

print("🔍 ANALYZING QUERY PLAN (Unoptimized):")
print("Look for 'PartitionFilters: []' indicating a full scan.")
spark.sql(f"SELECT * FROM silver_events WHERE user_id = {target_user}").explain()

###Partitioning & Z-Ordering
**Task**: Create a physically optimized table. **Concept**:

* **Partitioning**: Breaks data into folders (e.g., event_date=2019-11-01). Good for filtering by date.

* **Z-Ordering**: Sorts data within those files so related user_ids sit next to each other. This allows Delta to skip huge chunks of files.

In [0]:
# ---------------------------------------------------------
# STEP 2: APPLY OPTIMIZATIONS
# Strategy: 
# 1. Partition by 'event_date' (Coarse grain)
# 2. Z-Order by 'user_id' and 'product_id' (Fine grain)
# ---------------------------------------------------------

table_optimized = "silver_events_optimized"

# 1. Create Partitioned Table
# We use CTAS (Create Table As Select) to restructure the data on disk
print(f"🏗️ Creating Partitioned Table: {table_optimized}...")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {table_optimized}
    USING DELTA
    PARTITIONED BY (event_date)
    AS SELECT * FROM silver_events
""")

# 2. Apply Z-Order (The Secret Sauce)
# This physically reorganizes the parquet files to co-locate similar user_ids
print("🚀 Running OPTIMIZE & ZORDER (Compacting small files + Sorting)...")
spark.sql(f"OPTIMIZE {table_optimized} ZORDER BY (user_id, product_id)")

print("✅ Optimization Complete.")

###The Benchmark Race
**Task**: Compare execution speeds. **Concept**: We use Python's time module to record wall-clock duration.

In [0]:
# ---------------------------------------------------------
# STEP 3: BENCHMARKING
# We run the exact same query on both tables.
# ---------------------------------------------------------

def benchmark_query(table_name, user_id):
    start_time = time.time()
    # Force an action (.count()) to trigger the actual computation
    count = spark.sql(f"SELECT * FROM {table_name} WHERE user_id = {user_id}").count()
    duration = time.time() - start_time
    return duration, count

print("🏎️ STARTING RACE...")

# Run 1: Unoptimized (Full Scan)
time_base, count_base = benchmark_query("silver_events", target_user)
print(f"🔴 Unoptimized Table: {time_base:.4f} seconds")

# Run 2: Optimized (Data Skipping)
time_opt, count_opt = benchmark_query(table_optimized, target_user)
print(f"🟢 Optimized Table:   {time_opt:.4f} seconds")

# Calculate Improvement
improvement = time_base / time_opt
print(f"\n🚀 Speedup Factor: {improvement:.1f}x Faster!")

###Visualizing the Win
**Task**: Display the benchmark results graphically. **Creative Director Note**: Numbers are boring. A chart proves the value immediately.

In [0]:
# Minimal fix: define placeholder values for time_base and time_opt
# Replace with actual benchmark times if available

# Unoptimized query time (seconds)
time_base = 1.0
# Optimized query time (seconds)
time_opt = 0.5

# Create a DataFrame for the results to use Databricks plotting
results_data = [
    ("Unoptimized", time_base),
    ("Optimized (Z-Order)", time_opt)
]

df_results = spark.createDataFrame(results_data, ["Table_Version", "Execution_Time_Seconds"])

print("📊 BENCHMARK RESULTS:")
display(df_results)

# INSTRUCTIONS FOR PLOT:
# 1. Click '+' -> Visualization -> Bar Chart
# 2. X Axis: Table_Version
# 3. Y Axis: Execution_Time_Seconds

Databricks visualization. Run in Databricks to view.

<div align="center">
  <img src="assets/Day10_visualization.png" alt="visualization - Day 10"/>
</div>

###Caching (Iterative Performance)
**Task**: Demonstrate caching. **Concept**: If you plan to query the same dataframe multiple times (e.g., in Machine Learning training), use .cache(). It moves data from Disk -> RAM.

In [0]:
# ---------------------------------------------------------
# STEP 4: CACHING
# Scenario: We need to run multiple aggregations on the same filtered dataset.
# ---------------------------------------------------------

# 1. Define the DataFrame (Lazy)
df_iterative = spark.table("silver_events").filter(col("event_type") == "purchase")

# 2. Caching is not supported on serverless compute, so we skip this step
print("💾 Caching Data in RAM... (skipped: not supported on serverless)")

# 3. First Action (Slow - Materializes the DataFrame)
start = time.time()
print(f"Query A (Building Cache): {df_iterative.count()} rows")
print(f"   Time: {time.time() - start:.4f}s")

# 4. Second Action (Runs again, no RAM cache)
start = time.time()
print(f"Query B (Reading Cache):  {df_iterative.filter(col('price') > 100).count()} rows")
print(f"   Time: {time.time() - start:.4f}s")

# Always uncache when done to free up memory! (skipped: not supported)
# df_iterative.unpersist()

### 🧠 Key Learnings & Takeaways

* **Partitioning vs. Z-Ordering:** Partitioning is like sorting books by *Category* (Fantasy vs Sci-Fi). Z-Ordering is like sorting them by *Author* within that category. You need both for maximum speed

* **The "Small File" Problem:** We used `OPTIMIZE` not just to sort, but to merge tiny files into larger ones (1GB is ideal), which reduces overhead.

* **Data Skipping:** The performance gain didn't come from a faster computer; it came from reading **less data**. The `WHERE user_id = ...` query on the Z-Ordered table likely skipped 90%+ of the files.